In [1]:
import json
import pandas as pd
from dotenv import load_dotenv
import os
import re
from ast import literal_eval

from langchain_openai import ChatOpenAI
from langchain_core.globals import set_debug, set_verbose, set_llm_cache
from langchain_community.cache import InMemoryCache
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

import hashlib
import math
import threading
from queue import Queue
from typing import Iterable, List, Tuple, Set

import psycopg2
from psycopg2.pool import SimpleConnectionPool
from psycopg2 import extensions as _pg_ext
from psycopg2 import sql
from pgvector.psycopg2 import register_vector

from rdflib import Graph, URIRef, BNode, Literal
from langchain_openai import OpenAIEmbeddings

import threading
import numpy as np
import ast

In [2]:
load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")

In [3]:
DATASET_NAME = "hotpotQA"

BATCH_SIZE = 10
IS_DEBUG = True
DEBUG_QUESTION_SIZE = 500

# LLM_MODEL = "gpt-4.1-mini"
GRAPH_CREATION_LLM_MODEL = "gpt-4.1"
QUESTION_ENTITY_EXTRACTION_LLM_MODEL = "gpt-4.1"
QUESTION_ANSWERING_LLM_MODEL = "gpt-4.1"
EVALUATOR_LLM_MODEL = "gpt-4.1"

train_hotpot_qa_path = r'C:\Users\Sudheera\Documents\Phd\LLMs_and_KGs\hotpotQA\hotpot_train_v1.1.json'
test_hotpot_qa_path = r'C:\Users\Sudheera\Documents\Phd\LLMs_and_KGs\hotpotQA\hotpot_test_v1.1.json'

db_config = {
    'dbname': 'langchain',
    'user': 'langchain',
    'password': 'langchain',
    'host': 'localhost',
    'port': '6024'
}

In [4]:
OUTPUT_DIR = r'C:\Users\Sudheera\Documents\Phd\LLMs_and_KGs\hotpotQA_outputs'


KNOWLEDGE_GRAPH_PATH = rf'{OUTPUT_DIR}\{DATASET_NAME}\knowledge_graphs'
if not os.path.exists(KNOWLEDGE_GRAPH_PATH):
    os.makedirs(KNOWLEDGE_GRAPH_PATH)

MAIN_DF_AFTER_KG_CREATION_PATH = rf'{OUTPUT_DIR}\{DATASET_NAME}\main_df_after_kg_creation'
if not os.path.exists(MAIN_DF_AFTER_KG_CREATION_PATH):
    os.makedirs(MAIN_DF_AFTER_KG_CREATION_PATH)

MAIN_DF_AFTER_QUESTION_ENTITY_EXTRACTION_PATH = rf'{OUTPUT_DIR}\{DATASET_NAME}\main_df_after_question_entity_extraction'
if not os.path.exists(MAIN_DF_AFTER_QUESTION_ENTITY_EXTRACTION_PATH):
    os.makedirs(MAIN_DF_AFTER_QUESTION_ENTITY_EXTRACTION_PATH)

MAIN_DF_AFTER_QUESTION_ANSWERING_PATH = rf'{OUTPUT_DIR}\{DATASET_NAME}\main_df_after_question_answering'
if not os.path.exists(MAIN_DF_AFTER_QUESTION_ANSWERING_PATH):
    os.makedirs(MAIN_DF_AFTER_QUESTION_ANSWERING_PATH)

MAIN_DF_AFTER_EVALUATION_PATH = rf'{OUTPUT_DIR}\{DATASET_NAME}\main_df_after_evaluation'
if not os.path.exists(MAIN_DF_AFTER_EVALUATION_PATH):
    os.makedirs(MAIN_DF_AFTER_EVALUATION_PATH)

In [5]:
set_debug(False)
set_verbose(False)
set_llm_cache(InMemoryCache())

graph_creation_llm_model = ChatOpenAI(model=GRAPH_CREATION_LLM_MODEL, api_key=openai_api_key)
question_entity_extraction_llm_model = ChatOpenAI(model=QUESTION_ENTITY_EXTRACTION_LLM_MODEL, api_key=openai_api_key)
question_answering_llm_model = ChatOpenAI(model=QUESTION_ANSWERING_LLM_MODEL, api_key=openai_api_key)
evaluator_llm_model = ChatOpenAI(model=EVALUATOR_LLM_MODEL, api_key=openai_api_key)

In [6]:
with open(train_hotpot_qa_path, 'r', encoding='utf-8') as f:
    train_hotpot_qa_json = json.load(f)

with open(test_hotpot_qa_path, 'r', encoding='utf-8') as f:
    test_hotpot_qa_json = json.load(f)

In [7]:
# Dividing the dataset into batches

def divide_and_batch(train_json, batch_size):
    batch_context = []
    batch_question = []
    batch_answer = []

    for i in range(0, len(train_json), batch_size):
        batch = train_json[i:i + batch_size]
        context = []
        question = []
        answer = []

        for item in batch:
            context_text = ""
            j = 1
            for ctx in item['context']:
                context_text += f"Title {j} : {ctx[0]} \nParagraph {j} : {''.join(ctx[1])}\n"
                j += 1
            context.append(context_text)
            question.append(item['question'])
            answer.append(item['answer'])

        batch_context.append(context)
        batch_question.append(question)
        batch_answer.append(answer)

    return batch_context, batch_question, batch_answer

if IS_DEBUG:
    train_hotpot_qa_json = train_hotpot_qa_json[:DEBUG_QUESTION_SIZE]
    test_hotpot_qa_json = test_hotpot_qa_json[:DEBUG_QUESTION_SIZE]
    batch_context, batch_question, batch_answer = divide_and_batch(train_hotpot_qa_json, BATCH_SIZE)
else:
    test_hotpot_qa_json = test_hotpot_qa_json[:DEBUG_QUESTION_SIZE]
    batch_context, batch_question, batch_answer = divide_and_batch(test_hotpot_qa_json, BATCH_SIZE)

del train_hotpot_qa_json
del test_hotpot_qa_json

In [8]:
def sanitize_str(name):
    """
    Sanitize a name by removing extra spaces, apostrophes, and ensuring proper formatting.
    """
    # Replace apostrophes with empty string
    name = name.replace("'s", "s")
    name = name.replace("'", "")

    # Replace spaces with underscores
    name = name.replace(" ", "_")
    name = name.replace("–", "_")

    # Remove any other problematic characters
    name = re.sub(r'[^\w\.\-_]', '', name)

    # Prefix if starts with digit
    if re.match(r"^\d", name):
        name = f"n{name}"

    # If empty string, return something safe
    if not name:
        name = "unknown"

    return name


def sanitize_entities_and_relations(triple):
    """
    Sanitize entities and relations in a triple by removing extra spaces and ensuring proper formatting.
    """
    return (
        sanitize_str(triple[0].strip().lower()),  # Subject
        sanitize_str(triple[1].strip().lower()),  # Relation
        sanitize_str(triple[2].strip().lower())  # Object
    )


def parse_custom_triples(triples_str):
    triples = []
    for line in triples_str.strip().splitlines():
        line = line.strip()
        if not line:
            continue
        # Remove enclosing parentheses
        if line.startswith('(') and line.endswith(')'):
            line = line[1:-1]
        # Now, split ONLY on the first two commas
        parts = []
        remaining = line
        for _ in range(2):
            # Find the first comma
            idx = remaining.find(',')
            if idx == -1:
                break
            parts.append(remaining[:idx].strip())
            remaining = remaining[idx + 1:].strip()
        parts.append(remaining)
        if len(parts) == 3:
            triples.append(sanitize_entities_and_relations(tuple(parts)))
    return triples


def extract_sections(text, reasoning_pat, triples_pat):
    # Extract sections using regex
    extracted_reasoning = re.search(reasoning_pat, text, re.DOTALL)
    extracted_triples = re.search(triples_pat, text, re.DOTALL)

    # Clean reasoning sections
    reasoning_str = extracted_reasoning.group(1).strip() if extracted_reasoning else ""

    # Use the custom parser for triples
    triples_list = parse_custom_triples(extracted_triples.group(1)) if extracted_triples else []

    return reasoning_str, triples_list


def extract_context_to_columns(row):
    # Regex patterns for section headers
    context_reasoning_pat = r'### CONTEXT_REASONING\s*(.*?)\s*### CONTEXT_TRIPLES'
    context_triples_pat = r'### CONTEXT_TRIPLES\s*(.*)'

    context_reasoning, context_triples = extract_sections(row['graphs_str'], context_reasoning_pat, context_triples_pat)
    return pd.Series([context_reasoning, context_triples],
                     index=['context_reasoning', 'context_triples'])

In [9]:
from rdflib import Graph, Namespace, RDF, RDFS, OWL, URIRef
import os


def save_kg_from_triples(triples, batch_num, question_num):
    g = Graph()
    EX = Namespace("http://example.org/")

    # Add triples to the graph
    for subject, relation, obj in triples:
        subject_uri = URIRef(EX[subject])
        object_uri = URIRef(EX[obj])
        g.add((subject_uri, URIRef(EX[relation]), object_uri))

    # Define the file path
    file_path = rf"{KNOWLEDGE_GRAPH_PATH}\{DATASET_NAME}_ontology_b{batch_num}_q{question_num}.rdf"

    # Save the graph in OWL format
    g.serialize(destination=file_path, format='xml')
    print(f"Saved KG for batch {batch_num}, question {question_num} to {file_path}")

In [10]:
def extract_question_to_columns(row):
    # Regex patterns for section headers
    questions_reasoning_pat = r'### QUESTION_REASONING\s*(.*?)\s*### QUESTIONS_TRIPLES'
    questions_triples_pat = r'### QUESTIONS_TRIPLES\s*(.*)'

    questions_reasoning, questions_triples = extract_sections(row['questions_str'], questions_reasoning_pat,
                                                              questions_triples_pat)
    return pd.Series([questions_reasoning, questions_triples],
                     index=['question_reasoning', 'question_triples'])

In [11]:
# ------------------------------
# Connection pool + helpers
# ------------------------------
_DB_POOL = None
_DB_POOL_LOCK = threading.Lock()


def _get_pool() -> SimpleConnectionPool:
    global _DB_POOL
    if _DB_POOL is None:
        with _DB_POOL_LOCK:
            if _DB_POOL is None:
                _DB_POOL = SimpleConnectionPool(
                    minconn=1,
                    maxconn=16,  # adjust if you increase thread count
                    **db_config
                )
    return _DB_POOL


def _get_conn():
    pool = _get_pool()
    conn = pool.getconn()
    try:
        # Register pgvector adapter once per connection
        register_vector(conn)
    except Exception:
        pass
    _reset_conn(conn)
    return conn


def _put_conn(conn):
    try:
        _reset_conn(conn)  # return clean
    finally:
        _get_pool().putconn(conn)


def _reset_conn(conn):
    """Ensure the connection is not stuck in a failed or open transaction."""
    try:
        if conn.get_transaction_status() != _pg_ext.TRANSACTION_STATUS_IDLE:
            conn.rollback()
    except Exception:
        try:
            conn.rollback()
        except Exception:
            pass


# ------------------------------
# Schema management
# ------------------------------
def ensure_schema(embedding_dim: int = 1536):
    """
    - CREATE EXTENSION on a short-lived autocommit connection
    - CREATE TABLE/INDEXES on a pooled connection in a transaction
    """
    # 1) CREATE EXTENSION on its own connection (autocommit)
    try:
        ext_conn = psycopg2.connect(**db_config)
        ext_conn.autocommit = True
        with ext_conn.cursor() as cur:
            try:
                cur.execute("CREATE EXTENSION IF NOT EXISTS vector;")
            except psycopg2.Error as e:
                print("[ensure_schema] WARNING: CREATE EXTENSION vector failed.")
                print("[ensure_schema] pgerror:\n", getattr(e, "pgerror", str(e)))
                # Not fatal if already installed or if perms are restricted
    finally:
        try:
            ext_conn.close()
        except Exception:
            pass

    # 2) CREATE TABLE / INDEXES via pooled connection
    conn = _get_conn()
    try:
        with conn:
            with conn.cursor() as cur:
                cur.execute(sql.SQL("""
                    CREATE TABLE IF NOT EXISTS embeddings_cache (
                        id              BIGSERIAL PRIMARY KEY,
                        hash_word       TEXT        NOT NULL,
                        word            TEXT        NOT NULL,
                        word_embedding  vector({dim}) NOT NULL,
                        created_at      TIMESTAMPTZ NOT NULL DEFAULT now(),
                        updated_at      TIMESTAMPTZ NOT NULL DEFAULT now(),
                        UNIQUE (hash_word),
                        UNIQUE (word)
                    );
                """).format(dim=sql.Literal(embedding_dim)))

                # Attempt to align dimension (noop if same). May fail if existing data incompatible.
                try:
                    cur.execute(sql.SQL("""
                        ALTER TABLE embeddings_cache
                        ALTER COLUMN word_embedding TYPE vector({dim});
                    """).format(dim=sql.Literal(embedding_dim)))
                except psycopg2.Error as e:
                    print("[ensure_schema] NOTE: Could not alter word_embedding dimension.")
                    print("[ensure_schema] pgerror:\n", getattr(e, "pgerror", str(e)))

                cur.execute("""
                    CREATE INDEX IF NOT EXISTS idx_embeddings_cache_hash_word
                    ON embeddings_cache (hash_word);
                """)
    except psycopg2.Error as e:
        try:
            conn.rollback()
        except Exception:
            pass
        print("[ensure_schema] ERROR: DDL transaction failed and was rolled back.")
        print("[ensure_schema] pgerror:\n", getattr(e, "pgerror", str(e)))
        raise
    finally:
        _put_conn(conn)


# ---------- HASH HELPER ----------
def sha256_hex(s: str) -> str:
    return hashlib.sha256(s.encode("utf-8")).hexdigest()


# ------------------------------
# Cache ops (DB)
# ------------------------------
def fetch_embedding_from_db(word: str):
    """
    Returns list[float] if present, else None.
    """
    h = sha256_hex(word)
    conn = _get_conn()
    try:
        with conn.cursor() as cur:
            cur.execute(
                "SELECT word_embedding FROM embeddings_cache WHERE hash_word = %s",
                (h,)
            )
            row = cur.fetchone()
            return list(row[0]) if row else None
    finally:
        _put_conn(conn)


def upsert_embedding_in_db(word: str, embedding: List[float]):
    h = sha256_hex(word)
    conn = _get_conn()
    try:
        with conn:
            with conn.cursor() as cur:
                cur.execute(
                    """
                    INSERT INTO embeddings_cache (hash_word, word, word_embedding)
                    VALUES (%s, %s, %s)
                    ON CONFLICT (hash_word) DO UPDATE
                    SET word = EXCLUDED.word,
                        word_embedding = EXCLUDED.word_embedding,
                        updated_at = now();
                    """,
                    (h, word, embedding)
                )
    finally:
        _put_conn(conn)

def short_name(uri):
    s = str(uri)
    if '#' in s:
        return s.split('#')[-1]
    elif '/' in s:
        return s.split('/')[-1]
    return s

def get_or_create_embedding(word: str, embedder: OpenAIEmbeddings) -> List[float]:
    cached = fetch_embedding_from_db(word)
    if cached is not None:
        return cached
    emb = embedder.embed_query(clean_entity(word))
    upsert_embedding_in_db(word, emb)
    return emb

def cosine_similarity(vec1, vec2):
    v1 = np.array(vec1)
    v2 = np.array(vec2)
    if np.linalg.norm(v1) == 0 or np.linalg.norm(v2) == 0:
        return 0.0
    return float(np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2)))


def bfs_n_hop_triples(graph, start_entity, n):
    visited = set([start_entity])
    current_level = set([start_entity])
    neighborhood_triples = set()
    for hop in range(n):
        next_level = set()
        for node in current_level:
            # Outgoing
            for s, p, o in graph.triples((node, None, None)):
                next_level.add(o)
                neighborhood_triples.add((short_name(s), short_name(p), short_name(o)))
            # Incoming
            for s, p, o in graph.triples((None, None, node)):
                next_level.add(s)
                neighborhood_triples.add((short_name(s), short_name(p), short_name(o)))
        next_level -= visited
        if not next_level:
            break
        visited |= next_level
        current_level = next_level
    return list(neighborhood_triples)


def extract_entities(graph):
    entities = set()
    for s, p, o in graph:
        entities.add(s)
        entities.add(o)
    return list(entities)


def clean_entity(entity):
    """
    Cleans the entity string by removing unwanted characters and formatting.
    """
    entity = entity.replace('_', ' ')
    if entity.startswith('n') and entity[1].isdigit():
        entity = entity[1:]
    return entity.strip().lower()


def worker(entity_queue,
           graph,
           key,
           key_embedding,
           per,
           n,
           neighborhoods,
           matched_entities,
           embedder,
           thread_id: int = 0,  # <-- default so it won't crash if omitted
           lock: threading.Lock = None):
    print(f"[Thread {thread_id}] started.")
    while not entity_queue.empty():
        entity = entity_queue.get()
        try:
            # entity_label = short_name(entity)
            entity_label = entity
            # emb = get_embedding(entity_label, embedding_dict, lock, embedder)
            emb = get_or_create_embedding(short_name(entity), embedder)
            sim = cosine_similarity(key_embedding, emb)
            if sim >= per:
                print(f"similarity for '{key}' and '{short_name(entity)}' is {sim:.4f}")
                print(f"Extracting {n}-hop triples for entity '{entity}'")
                triples = bfs_n_hop_triples(graph, entity, n)
                with lock:
                    neighborhoods.extend(triples)
                    matched_entities.append(entity_label)
        except Exception as e:
            print(f"[Thread {thread_id}] Error processing entity {entity}: {e}")
        finally:
            entity_queue.task_done()
    print(f"[Thread {thread_id}] finished.")

In [12]:
# ---------- MAIN FUNCTION (DB-backed cache) ----------
def kg_neighborhood_extractor(
        rdf_path,
        per,
        n,
        key,
        k=4,
        embedding_model="text-embedding-3-small",
        embedding_dim=1536  # keep in sync with the model you use
):
    """
    Args:
        rdf_path: path to RDF/XML/Turtle/etc. file supported by rdflib
        per: cosine similarity threshold (0..1) to treat an entity as a match
        n: number of BFS hops for neighborhood expansion
        key: search string to embed & compare with entity labels
        k: number of worker threads
        embedding_model: OpenAI embedding model name (via LangChain)
        embedding_dim: pgvector column dimension; must match the model in use
    """
    print("Ensuring DB schema...")
    ensure_schema(embedding_dim=embedding_dim)

    print("Loading RDF graph...")
    graph = Graph()
    graph.parse(rdf_path)

    print("Extracting entities...")
    entities = extract_entities(graph)
    print(f"Total entities found: {len(entities)}")

    neighborhoods: List[Tuple[str, str, str]] = []
    matched_entities: List[Tuple[str, float]] = []
    lock = threading.Lock()

    print(f"Initializing LangChain OpenAIEmbeddings with model '{embedding_model}'...")
    embedder = OpenAIEmbeddings(model=embedding_model)

    print(f"Getting embedding for search key: '{key}'")
    key_embedding = get_or_create_embedding(key, embedder)

    # Queue up work
    entity_queue = Queue()
    for ent in entities:
        entity_queue.put(ent)

    # Threads
    threads = []
    for i in range(k):
        t = threading.Thread(
            target=worker,
            args=(
                entity_queue,
                graph,
                key,
                key_embedding,
                per,
                n,
                neighborhoods,
                matched_entities,
                embedder,
                i + 1,  # <-- thread_id
                lock,  # <-- lock
            ),
            daemon=True,
        )
        t.start()
        threads.append(t)

    # Wait
    entity_queue.join()
    for t in threads:
        t.join()

    # Deduplicate triples
    neighborhoods = list({tr for tr in neighborhoods})

    # Optional: show cache size
    cached_count = None
    conn = _get_conn()
    try:
        with conn.cursor() as cur:
            cur.execute("SELECT COUNT(*) FROM embeddings_cache;")
            cached_count = cur.fetchone()[0]
    finally:
        _put_conn(conn)

    print("\nSummary:")
    print(f"Matched entities (label, similarity): {matched_entities}")
    print(f"Total unique triples in neighborhoods: {len(neighborhoods)}")
    if cached_count is not None:
        print(f"Embeddings cached in DB: {cached_count}")

    return neighborhoods

In [13]:
def format_reference_triples(reference_triples):
    formatted_triples = []
    for triple in reference_triples:
        subject, relation, obj = triple
        subject = subject.replace('_', ' ')
        relation = relation.replace('_', ' ')
        obj = obj.replace('_', ' ')
        if subject.startswith('n') and subject[1].isdigit():
            subject = subject[1:]
        if obj.startswith('n') and obj[1].isdigit():
            obj = obj[1:]
        formatted_triples.append((subject, relation, obj))
    return formatted_triples

In [14]:
def extract_final_answer(row):
    # Regex pattern to find the final answer
    final_answer_pat = r'### FINAL_ANSWER\s*(.*)'
    match = re.search(final_answer_pat, row['llm_answer_str'], re.DOTALL)
    return match.group(1).strip() if match else "No answer found"

def extract_is_correct(row):
    # Regex pattern to find the final answer
    final_answer_pat = r'### FINAL_ANSWER\s*(TRUE|FALSE)'
    match = re.search(final_answer_pat, row['answer_check_str'], re.DOTALL)
    return match.group(1).strip() if match else False

In [15]:
graph_creation_system_msg = """
<role>
    You are an expert in natural language understanding and knowledge graph construction. Your task is to extract structured relationship triples from a context set (Titles and Paragraphs), including both explicit and logically inferable facts.
</role>

<behavior>
    <rule name="Entity Extraction">
        Identify and extract unique entities from Titles and Paragraphs. Use canonical forms for names. Capture entities including people, organizations, locations, publications, dates, and events.
    </rule>
    <rule name="Relationship Extraction">
        Extract explicit and clearly implied relationships between entities in the form of (subject, relation, object) triples.
        This includes:
        - Temporal relations such as publication or operational time spans (e.g., "published during", "active from", "merged in").
        - Authorship, editorial, or contribution roles (e.g., "edited by", "featured work by").
        - Event-based or structural relations (e.g., "merged into", "founded by", "took over").
    </rule>
    <rule name="Completeness">
        Extract **as many valid and informative triples as possible**. Include relationships that are inferred logically but unambiguous in context (e.g., date ranges, geographic associations, contributor roles).
    </rule>
    <rule name="Formatting and Output">
        Output must contain two sections, in order:
        1. ### CONTEXT_REASONING — step-by-step reasoning describing how entities and relationships were selected or inferred.
        2. ### CONTEXT_TRIPLES — list all extracted triples, each on its own line in the format (entity_1, relationship, entity_2)
        Do not add explanations, numbering, or extra text outside of these sections.
    </rule>
</behavior>

<format>
1. Carefully read the <context> section.
2. Begin with this heading:
   ### CONTEXT_REASONING
   Then, write concise reasoning steps describing how entities and relationships were selected and interpreted.
3. After reasoning, print this heading:
   ### CONTEXT_TRIPLES
   Then list each extracted triple on a new line in the format (entity_1, relationship, entity_2).
4. Do not include any explanations, text, or comments after the last triple.
</format>
"""

graph_creation_human_msg = """
<context>
{context}
</context>
"""

graph_creation_prompt = ChatPromptTemplate([("system", graph_creation_system_msg), ("human", graph_creation_human_msg)])
graph_creation_chain = graph_creation_prompt | graph_creation_llm_model | StrOutputParser()

In [16]:
question_entity_extraction_system_msg = """
<role>
    You are an expert in natural language understanding and knowledge graph construction. Your task is to convert a provided question into a set of question triples, using simple variable names (such as x, y, z, i, j, k) for unknown entities or values to be inferred.
</role>

<behavior>
    <rule name="Entity Extraction">
        Identify and extract unique entities. Use canonical names.
    </rule>
    <rule name="Relationship Extraction">
        Extract explicit or clear implicit relationships between entities as (subject, relation, object) triples.
    </rule>
    <rule name="Temporal and Comparative Relations">
        Use relations like "started in", "founded by", "wrote a song about", "is named after", etc., matching the logic and semantics in the context or question.
    </rule>
    <rule name="Variable Placeholders for Unknowns">
        For each unknown or answer to be found in the question, use a simple English variable (x, y, z, i, j, k, etc.) in place of the answer within the triples. Each distinct unknown in the question gets a unique variable.
    </rule>
    <rule name="Question Triple Decomposition">
        For each question, decompose it into one or more triples, using variable placeholders for any unknowns to be inferred from the context.
        Examples:
        - "Which magazine was started first Arthur's Magazine or First for Women?"
           (Arthur's Magazine, started in, x)
           (First for Women, started in, y)
        - "The Oberoi family is part of a hotel company that has a head office in what city?"
           (Oberoi family, is part of, x)
           (x, has head office in, y)
    </rule>
    <rule name="Formatting and Output">
        Output must contain four sections, in order:
        1. ### QUESTION_REASONING — a step-by-step reasoning of how the question was decomposed into question triples
        2. ### QUESTIONS_TRIPLES — list the question triples, each on its own line. each on its own line in the format (entity_1, relationship, entity_2)
        Do not add explanations, numbering, or extra text.
    </rule>
</behavior>

<format>
1. Carefully read the <question> section.
2. Before extracting question triples, explain step by step how you break down the question and assign variable placeholders. Print this heading:
   ### QUESTION_REASONING
   Then write your reasoning in clear, concise steps.
3. After reasoning, print the question triples. Print this heading:
   ### QUESTIONS_TRIPLES
   List each question triple on a new line, using variable placeholders (x, y, z, etc.) for unknowns.
4. After the last question triple, end with no extra text or parentheses.
</format>
"""

question_entity_extraction_human_msg = """
<question>
{question}
</question>
"""

question_entity_extraction_prompt = ChatPromptTemplate([("system", question_entity_extraction_system_msg), ("human", question_entity_extraction_human_msg)])
question_entity_extraction_chain = question_entity_extraction_prompt | question_entity_extraction_llm_model | StrOutputParser()

In [17]:
question_answering_system_msg = """
<role>
    You are a logical reasoning expert tasked with answering questions based only on provided structured knowledge in the form of Reference Triples.
</role>

<behavior>
    <rule name="Step-by-Step Reasoning">
        Use the reference triples to deduce relevant facts in a step-by-step manner. Clearly identify which triples are used for each inference.
    </rule>
    <rule name="Comparative Reasoning">
        When comparing entities, extract or infer comparable attributes (e.g., start dates, founding years) and use them to conclude which comes first, is larger, etc., as applicable.
    </rule>
    <rule name="Answer Reporting">
        At the end of your reasoning, provide the final answer on a new line prefixed with "### FINAL_ANSWER", followed by the concise answer on the next line.
        If the data is insufficient to answer the question definitively, return "insufficient data".
    </rule>
</behavior>

<format>
1. Carefully examine the <question> and <reference_triples>.
2. Begin with step-by-step reasoning based strictly on the reference triples.
3. End your output with the heading "### FINAL_ANSWER" followed by the answer on the next line.
</format>
"""

question_answering_human_msg = """
<question>
{question}
</question>

<reference triples>
{reference_triples}
</reference triples>
"""

question_answering_prompt = ChatPromptTemplate([("system", question_answering_system_msg), ("human", question_answering_human_msg)])
question_answering_chain = question_answering_prompt | question_answering_llm_model | StrOutputParser()

In [18]:
evaluator_system_msg = """
<role>
    You are a meticulous LLM answer evaluator. Your task is to determine if the provided llm_answer correctly matches the correct_answer for a given question.
</role>

<behavior>
    <rule name="case-insensitive-match">
        Treat answers as correct even if their case (uppercase/lowercase) does not match, as long as the content matches.
    </rule>
    <rule name="synonyms-acceptable">
        Accept synonyms, short forms, or equivalent factual answers (e.g., "alcohol" and "ethanol") as correct, unless there is a clear difference in meaning or context.
    </rule>
    <rule name="factual-containment">
        Accept answers that contain the correct answer as a substring, or are paraphrased, provided no contradictory information is introduced.
    </rule>
    <rule name="insufficient-data">
        Mark as incorrect if llm_answer indicates "insufficient data" or similar phrases, but a correct factual answer is actually provided in correct_answer.
    </rule>
    <rule name="incorrect-content">
        Mark as incorrect if the llm_answer gives a wrong, contradictory, or irrelevant response compared to correct_answer.
    </rule>
    <rule name="format-neutrality">
        Do not penalize for minor differences in format, such as punctuation or extra explanatory words, as long as the meaning is unchanged.
    </rule>
    <rule name="numerical-tolerance">
        For numerical answers, accept equivalent formats (e.g., "2006" and "September 2006" are correct if both indicate the correct year), but mark as incorrect if the core value is wrong.
    </rule>
</behavior>

<format>
1) Output your reasoning step by step, referencing the question, correct_answer, and llm_answer.
2) Clearly state if the llm_answer matches the correct_answer according to the behavior rules above.
3) End your output with the heading "### FINAL_ANSWER" followed by TRUE if the llm_answer is correct or FALSE if incorrect.
</format>
"""

evaluator_human_msg = """
<question>
{question}
</question>

<correct_answer>
{correct_answer}
</correct_answer>

<llm_answer>
{llm_answer}
</llm_answer>
"""

evaluator_prompt = ChatPromptTemplate([("system", evaluator_system_msg), ("human", evaluator_human_msg)])
evaluator_chain = evaluator_prompt | evaluator_llm_model | StrOutputParser()

In [20]:
total_batches = len(batch_context)

for i in range(total_batches):
    main_df = pd.DataFrame()
    main_df_after_kg_creation_csv_name = rf"{MAIN_DF_AFTER_KG_CREATION_PATH}\{DATASET_NAME}_main_df_after_kg_creation_b{i}.csv"
    main_df_after_question_entity_extraction_csv_name = rf"{MAIN_DF_AFTER_QUESTION_ENTITY_EXTRACTION_PATH}\{DATASET_NAME}_main_df_after_question_entity_extraction_b{i}.csv"
    main_df_after_question_answering_csv_name = rf"{MAIN_DF_AFTER_QUESTION_ANSWERING_PATH}\{DATASET_NAME}_main_df_after_question_answering_b{i}.csv"
    main_df_after_evaluation_csv_name = rf"{MAIN_DF_AFTER_EVALUATION_PATH}\{DATASET_NAME}_main_df_after_evaluation_b{i}.csv"

    print(f"Processing batch {i + 1}/{total_batches} ...")

    # If main_df_after_kg_creation_csv_name exists, load it
    if os.path.exists(main_df_after_kg_creation_csv_name):
        print(f"Loading existing DataFrame from {main_df_after_kg_creation_csv_name}")
        main_df = pd.read_csv(main_df_after_kg_creation_csv_name)
        print("-" * 50)
    else:
        batch_num_list = [i] * len(batch_context[i])
        question_num_list = list(range(1, len(batch_context[i]) + 1))
        context_list = []
        question_list = []
        answers_list = []

        graph_creation_batch_input_batch = []
        j = 0
        for batch in range(len(batch_context[i])):
            context = batch_context[i][batch]
            graph_creation_batch_input_batch.append({
                "context": context
            })
            context_list.append(context)
            question_list.append(batch_question[i][j])
            answers_list.append(batch_answer[i][j])
            j += 1

        df = pd.DataFrame({
            "batch_num": batch_num_list,
            "question_num": question_num_list,
            "context": context_list,
            "question": question_list,
            "answer": answers_list
        })
        main_df = pd.concat([main_df, df], ignore_index=True)

        j = 0
        for item in graph_creation_batch_input_batch:
            print("Context:", item['context'])
            print("-" * 50)
            j += 1
            if j >= 5:
                print("=" * 50)
                break

        # Process the batch using the graph creation chain
        print(f"Generating Graphs for Batch {i + 1} ... processing {len(graph_creation_batch_input_batch)} items")
        graphs_str_list = graph_creation_chain.batch(graph_creation_batch_input_batch)

        print(f"Batch {i + 1} ... processed {len(graphs_str_list)} items")
        for j, graph_str in enumerate(graphs_str_list):
            print(f"Item {j + 1}:")
            print(graph_str)
            print("-" * 50)
            if j >= 5:
                print("=" * 50)
                break

        # Adding the generated graphs to the main DataFrame
        graphs_str_df = pd.DataFrame({
            "batch_num": [],
            "question_num": [],
            "graphs_str": []
        })
        batch_num_list = [i] * len(graphs_str_list)
        question_num_list = list(range(1, len(graphs_str_list) + 1))
        graphs_str_df = pd.concat([graphs_str_df, pd.DataFrame({
            "batch_num": batch_num_list,
            "question_num": question_num_list,
            "graphs_str": graphs_str_list
        })], ignore_index=True)

        del graphs_str_list
        del graph_creation_batch_input_batch
        # del batch_context
        # del batch_question
        # del batch_answer

        # Extracting context reasoning and triples from the generated graphs
        graphs_str_df[['context_reasoning', 'context_triples']] = graphs_str_df.apply(extract_context_to_columns, axis=1)
        main_df = main_df.merge(graphs_str_df, on=['batch_num', 'question_num'], how='left')

        del graphs_str_df

        # Save the knowledge graph from the context triples
        print(f"Saving Knowledge Graphs for Batch {i + 1} ...")
        print(f"Knowledge Graphs will be saved to: {KNOWLEDGE_GRAPH_PATH}")
        if not os.path.exists(KNOWLEDGE_GRAPH_PATH):
            os.makedirs(KNOWLEDGE_GRAPH_PATH)

        for index, row in main_df.iterrows():
            batch_num = row['batch_num']
            question_num = row['question_num']
            context_triples = row['context_triples']

            if context_triples:  # Only save if there are context triples
                save_kg_from_triples(context_triples, batch_num, question_num)

        # Save the main DataFrame to a CSV file
        main_df.to_csv(main_df_after_kg_creation_csv_name, index=False)
        print(rf"Main DataFrame for Batch {i + 1} saved to {MAIN_DF_AFTER_KG_CREATION_PATH}\{DATASET_NAME}_main_df_after_kg_creation_b{i}.csv")

    ##########################################################################################################################################################

    # If main_df_after_question_entity_extraction_csv_name exists, load it
    if os.path.exists(main_df_after_question_entity_extraction_csv_name):
        print(f"Loading existing DataFrame from {main_df_after_question_entity_extraction_csv_name}")
        main_df = pd.read_csv(main_df_after_question_entity_extraction_csv_name)
        print("-" * 50)
    else:
        # Extracting Entities from Questions
        print("Extracting Entities from Questions ...")
        question_entity_extraction_input_item = []

        for j, (index, row) in enumerate(main_df.iterrows()):
            question_entity_extraction_input_item.append({
                "question": row['question']
            })

        j = 0
        for item in question_entity_extraction_input_item:
            print("Question:", item['question'])
            print("-" * 50)
            j += 1
            if j >= 5:
                print("=" * 50)
                break

        print(f"Batch {i + 1} ... processing {len(question_entity_extraction_input_item)} items")
        question_entity_extraction_str_list = question_entity_extraction_chain.batch(question_entity_extraction_input_item)

        print(f"Batch {i + 1} ... processed {len(question_entity_extraction_str_list)} items")
        for j, graph_str in enumerate(question_entity_extraction_str_list):
            print(f"Item {j + 1}:")
            print(graph_str)
            print("-" * 50)

        questions_str_df = pd.DataFrame({
            "batch_num": [],
            "question_num": [],
            "questions_str": []
        })
        batch_num_list = [i] * len(question_entity_extraction_str_list)
        question_num_list = list(range(1, len(question_entity_extraction_str_list) + 1))
        questions_str_df = pd.concat([questions_str_df, pd.DataFrame({
            "batch_num": batch_num_list,
            "question_num": question_num_list,
            "questions_str": question_entity_extraction_str_list
        })], ignore_index=True)

        questions_str_df[['question_reasoning', 'question_triples']] = questions_str_df.apply(extract_question_to_columns, axis=1)
        main_df = main_df.merge(questions_str_df, on=['batch_num', 'question_num'], how='left')

        # Save the main DataFrame after question entity extraction
        main_df.to_csv(main_df_after_question_entity_extraction_csv_name, index=False)
        print(rf"Main DataFrame after Question Entity Extraction for Batch {i + 1} saved to {MAIN_DF_AFTER_QUESTION_ENTITY_EXTRACTION_PATH}\{DATASET_NAME}_main_df_after_question_entity_extraction_b{i}.csv")

        del question_entity_extraction_input_item
        del question_entity_extraction_str_list
        del questions_str_df

    ##########################################################################################################################################################

    # if main_df_after_question_answering_csv_name exists, load it
    if os.path.exists(main_df_after_question_answering_csv_name):
        print(f"Loading existing DataFrame from {main_df_after_question_answering_csv_name}")
        main_df = pd.read_csv(main_df_after_question_answering_csv_name)
        print("-" * 50)
    else:
        # Extracting neighborhoods for each question
        neighborhood_extraction_input_item = []
        for j, (index, row) in enumerate(main_df.iterrows()):
            reference_triples = []
            for triple in row['question_triples']:
                # Extract reference_triples using question_triples
                if len(triple[0]) > 1:
                    reference_triples += kg_neighborhood_extractor(
                        rdf_path=rf"{KNOWLEDGE_GRAPH_PATH}\{DATASET_NAME}_ontology_b{row['batch_num']}_q{row['question_num']}.rdf",
                        per=0.75, n=3, key=triple[0], k=3, embedding_model="text-embedding-3-small"
                    )
                print("-" * 50)
                if len(triple[2]) > 1:
                    reference_triples += kg_neighborhood_extractor(
                        rdf_path=rf"{KNOWLEDGE_GRAPH_PATH}\{DATASET_NAME}_ontology_b{row['batch_num']}_q{row['question_num']}.rdf",
                        per=0.75, n=3, key=triple[2], k=3, embedding_model="text-embedding-3-small"
                    )

            neighborhood_extraction_input_item.append({
                "question": row['question'],
                "reference_triples": reference_triples
            })
        print("=" * 50)

        # Formatting the reference triples
        print("Formatting reference triples ...")
        for j, item in enumerate(neighborhood_extraction_input_item):
            item['reference_triples'] = format_reference_triples(item['reference_triples'])

        # Process the batch using the question answering chain
        print(f"Generating Answers for Batch {i + 1} ... processing {len(neighborhood_extraction_input_item)} items")
        question_answering_str_list = question_answering_chain.batch(neighborhood_extraction_input_item)

        print(f"Batch {i + 1} ... processed {len(question_answering_str_list)} items")
        for j, answer_str in enumerate(question_answering_str_list):
            print(f"Item {j + 1}:")
            print(answer_str)
            print("-" * 50)
            if j >= 5:
                print("=" * 50)
                break

        # Adding the generated answers to the main DataFrame
        llm_answer_str_df = pd.DataFrame({
            "batch_num": [],
            "question_num": [],
            "llm_answer_str": []
        })

        batch_num_list = [i] * len(question_answering_str_list)
        question_num_list = list(range(1, len(question_answering_str_list) + 1))
        llm_answer_str_df = pd.concat([llm_answer_str_df, pd.DataFrame({
            "batch_num": batch_num_list,
            "question_num": question_num_list,
            "llm_answer_str": question_answering_str_list
        })], ignore_index=True)

        llm_answer_str_df['llm_answer'] = llm_answer_str_df.apply(extract_final_answer, axis=1)

        # Merging the answers into the main DataFrame
        main_df = main_df.merge(llm_answer_str_df, on=['batch_num', 'question_num'], how='left')

        # Save the main DataFrame after question answering
        main_df.to_csv(main_df_after_question_answering_csv_name, index=False)
        print(rf"Main DataFrame after Question Answering for Batch {i + 1} saved to {MAIN_DF_AFTER_QUESTION_ANSWERING_PATH}\{DATASET_NAME}_main_df_after_question_answering_b{i}.csv")

        del llm_answer_str_df
        del question_answering_str_list
        del neighborhood_extraction_input_item

    ##########################################################################################################################################################

    # If main_df_after_evaluation_csv_name exists, load it
    if os.path.exists(main_df_after_evaluation_csv_name):
        print(f"Loading existing DataFrame from {main_df_after_evaluation_csv_name}")
        main_df = pd.read_csv(main_df_after_evaluation_csv_name)
        print("-" * 50)

        accuracy = main_df['is_correct'].value_counts(normalize=True).get(True, 0) * 100
        print(f"Accuracy: {accuracy:.2f}%")
    else:
        # Evaluating the answers
        evaluator_input_item = []
        for j, (index, row) in enumerate(main_df.iterrows()):
            evaluator_input_item.append({
                "question": row['question'],
                "correct_answer": row['answer'],
                "llm_answer": row['llm_answer']
            })

        print(f"Evaluating Answers for Batch {i + 1} ... processing {len(evaluator_input_item)} items")
        j = 0
        for index, row in main_df.iterrows():
            print("Question:", row['question'])
            print("Correct Answer:", row['answer'])
            print("LLM Answer:", row['llm_answer'])
            print("-" * 50)
            j += 1
            if j >= 5:
                print("=" * 50)
                break

        print(f"Batch {i + 1} ... evaluating {len(evaluator_input_item)} items")
        answer_check_str_list = evaluator_chain.batch(evaluator_input_item)

        print(f"Batch {i + 1} ... evaluated {len(answer_check_str_list)} items")
        for j, answer_check_str in enumerate(answer_check_str_list):
            print(f"Item {j + 1}:")
            print(answer_check_str)
            print("-" * 50)
            if j >= 5:
                print("=" * 50)
                break

        answer_check_df = pd.DataFrame({
            "batch_num": [],
            "question_num": [],
            "answer_check_str": []
        })
        question_num_list = list(range(1, len(answer_check_str_list) + 1))
        answer_check_df = pd.concat([answer_check_df, pd.DataFrame({
            "batch_num": [i] * len(question_num_list),
            "question_num": question_num_list,
            "answer_check_str": answer_check_str_list
        })], ignore_index=True)

        # Merge the llm_answer_str_df with main_df to get the final answers
        answer_check_df['is_correct'] = answer_check_df.apply(extract_is_correct, axis=1)
        main_df = main_df.merge(answer_check_df, on=['batch_num', 'question_num'], how='left')

        # Save the main DataFrame after evaluation
        main_df.to_csv(main_df_after_evaluation_csv_name, index=False)
        print(rf"Main DataFrame after Evaluation for Batch {i + 1} saved to {MAIN_DF_AFTER_EVALUATION_PATH}\{DATASET_NAME}_main_df_after_evaluation_b{i}.csv")

        accuracy = main_df['is_correct'].value_counts(normalize=True).get('TRUE', 0) * 100
        print(f"Accuracy: {accuracy:.2f}%")

        del answer_check_df
        del evaluator_input_item
        del answer_check_str_list

    del main_df

    print(f"Batch {i + 1} completed.\n")
    print("=" * 50)
    print("=" * 50)
    print('\n' * 2)

Processing batch 1/50 ...
Loading existing DataFrame from C:\Users\Sudheera\Documents\Phd\LLMs_and_KGs\hotpotQA_outputs\hotpotQA\main_df_after_kg_creation\hotpotQA_main_df_after_kg_creation_b0.csv
--------------------------------------------------
Loading existing DataFrame from C:\Users\Sudheera\Documents\Phd\LLMs_and_KGs\hotpotQA_outputs\hotpotQA\main_df_after_question_entity_extraction\hotpotQA_main_df_after_question_entity_extraction_b0.csv
--------------------------------------------------
Loading existing DataFrame from C:\Users\Sudheera\Documents\Phd\LLMs_and_KGs\hotpotQA_outputs\hotpotQA\main_df_after_question_answering\hotpotQA_main_df_after_question_answering_b0.csv
--------------------------------------------------
Loading existing DataFrame from C:\Users\Sudheera\Documents\Phd\LLMs_and_KGs\hotpotQA_outputs\hotpotQA\main_df_after_evaluation\hotpotQA_main_df_after_evaluation_b0.csv
--------------------------------------------------
Accuracy: 90.00%
Batch 1 completed.




Proc

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

In [25]:
# Read all CSV files in MAIN_DF_AFTER_EVALUATION_PATH and concatenate them into a single DataFrame. Need 'batch_num', 'question_num', 'question', 'answer', 'llm_answer', 'is_correct' columns only.
final_df = pd.DataFrame()
for file in os.listdir(MAIN_DF_AFTER_EVALUATION_PATH):
    if file.endswith(".csv") and file.startswith(f"{DATASET_NAME}_main_df_after_evaluation_b"):
        file_path = os.path.join(MAIN_DF_AFTER_EVALUATION_PATH, file)
        df = pd.read_csv(file_path, usecols=['batch_num', 'question_num', 'question', 'answer', 'llm_answer', 'is_correct'])
        final_df = pd.concat([final_df, df], ignore_index=True)
final_df

,batch_num,question_num,question,answer,llm_answer,is_correct
0,0,1,Which magazine was started first Arthur's Maga...,Arthur's Magazine,Arthur's Magazine was started first.,True
1,0,2,The Oberoi family is part of a hotel company t...,Delhi,Delhi,True
2,0,3,Musician and satirist Allie Goertz wrote a son...,President Richard Nixon,Richard Nixon's middle name,True
3,0,4,What nationality was James Henry Miller's wife?,American,american,True
4,0,5,Cadmium Chloride is slightly soluble in this c...,alcohol,"ethanol, ethyl alcohol, drinking alcohol",True
...,...,...,...,...,...,...
325,9,6,Who invented the type of script used in autogr...,the Sumerians,insufficient data,False
326,9,7,Approximately what percentage of the global po...,17%,insufficient data,False
327,9,8,The Boren-McCurdy proposals were partially bro...,David Lyle Boren,senator david boren,True
328,9,9,The Thoen Stone is on display at a museum in w...,Lawrence County,insufficient data,False


In [27]:
# Calculate overall accuracy
overall_accuracy = final_df['is_correct'].value_counts(normalize=True).get(True, 0) * 100
print(f"Overall Accuracy: {overall_accuracy:.2f}%")

Overall Accuracy: 82.12%


In [28]:
# Number of questions
total_questions = final_df.shape[0]
print(f"Total Questions: {total_questions}")

Total Questions: 330
